# Treino e avaliacao comparativa dos cinco algoritmos para predicao de obito em chikungunya

Este notebook treina e avalia, num unico fluxo diretamente comparavel, os cinco algoritmos considerados para a predicao de obito em casos internados por chikungunya: Regressao Logistica, Arvore de Decisao (CART), Random Forest, XGBoost e LightGBM.
Para cada algoritmo, o pre-processamento, a busca de hiperparametros e a avaliacao seguem o mesmo protocolo, o que torna a comparacao final entre os cinco modelos direta e justa.
Para cada algoritmo, sao treinadas e comparadas duas versoes, uma versao padrao com os hiperparametros default da propria biblioteca e uma versao otimizada por busca em grade, e a versao usada no restante do fluxo e a de maior AUPRC media na validacao cruzada do treino.
A busca de hiperparametros usa validacao temporal por expanding window dentro do periodo de treino, de 2020 a 2023, o treino final usa todo o periodo de treino, a avaliacao principal usa o ano de 2024, e as predicoes sobre o ano de 2025 sao geradas e salvas, mas nao usadas para escolher o modelo.
Todas as funcoes de pre-processamento, validacao temporal e avaliacao ja estao implementadas em `utils/preprocessing.py` e sao reutilizadas aqui, sem reimplementacao.

Importa as bibliotecas necessarias, inclui a raiz do projeto no caminho de importacao para reutilizar o modulo `utils/preprocessing.py`, e cria os diretorios de saida usados ao longo do notebook.
As predicoes, os modelos e os resultados da busca de hiperparametros sao salvos em `output/`, enquanto a tabela comparativa final e salva em `analysis/results_modelagem/`.

In [1]:
import json
import sys
from pathlib import Path

import numpy as np
import pandas as pd
from lightgbm import LGBMClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import GridSearchCV
from sklearn.pipeline import Pipeline
from sklearn.tree import DecisionTreeClassifier
from xgboost import XGBClassifier

PROJECT_ROOT = Path.cwd().parent.parent
sys.path.insert(0, str(PROJECT_ROOT))

from utils import preprocessing

DIRETORIO_METRICAS = PROJECT_ROOT / "output" / "metricas"
DIRETORIO_MODELOS = PROJECT_ROOT / "output" / "modelos"
DIRETORIO_RESULTADOS_ANALISE = PROJECT_ROOT / "analysis" / "results_modelagem"

DIRETORIO_METRICAS.mkdir(parents=True, exist_ok=True)
DIRETORIO_MODELOS.mkdir(parents=True, exist_ok=True)
DIRETORIO_RESULTADOS_ANALISE.mkdir(parents=True, exist_ok=True)

Carrega os conjuntos de preditores e alvo ja construidos pelo pipeline de dados: treino de 2020 a 2023, teste de 2024 e validacao externa de 2025.
O alvo e extraido como serie, ja que cada arquivo de alvo contem uma unica coluna, `obito`.

In [2]:
DIRETORIO_FEATURES = PROJECT_ROOT / "data" / "features" / "baseline"

X_train = pd.read_parquet(DIRETORIO_FEATURES / "X_train.parquet")
y_train = pd.read_parquet(DIRETORIO_FEATURES / "y_train.parquet")[preprocessing.COLUNA_ALVO]

X_test = pd.read_parquet(DIRETORIO_FEATURES / "X_test.parquet")
y_test = pd.read_parquet(DIRETORIO_FEATURES / "y_test.parquet")[preprocessing.COLUNA_ALVO]

X_val2025 = pd.read_parquet(DIRETORIO_FEATURES / "X_val2025.parquet")
y_val2025 = pd.read_parquet(DIRETORIO_FEATURES / "y_val2025.parquet")[preprocessing.COLUNA_ALVO]

X_train.shape, X_test.shape, X_val2025.shape

((7171, 28), (3978, 28), (3241, 28))

Reconstroi os tres folds de validacao temporal por expanding window a partir de `data/processed/chik_treated.parquet`, alinhados posicionalmente com `X_train` e `y_train`.
O primeiro fold treina em 2020 e valida em 2021, o segundo treina em 2020 e 2021 e valida em 2022, e o terceiro treina em 2020 a 2022 e valida em 2023.
Esses folds sao usados pelo `GridSearchCV` dos cinco algoritmos, para que a busca de hiperparametros nunca use dados do proprio ano de validacao nem de anos futuros a ele.

In [3]:
folds_expanding_window = preprocessing.construir_folds_expanding_window(
    PROJECT_ROOT, tamanho_treino=len(X_train)
)

[(len(indices_treino), len(indices_validacao)) for indices_treino, indices_validacao in folds_expanding_window]

[(936, 1262), (2198, 2532), (4730, 2441)]

Separa as colunas numericas das categoricas, segundo a mesma convencao usada em `utils/preprocessing.py`, e calcula o peso de classe positiva usado por XGBoost e LightGBM a partir da prevalencia de obito no treino completo.
Para Regressao Logistica, Arvore de Decisao e Random Forest, o desbalanceamento e tratado pelo parametro `class_weight="balanced"` do proprio scikit-learn, diretamente na secao de cada algoritmo.

In [4]:
COLUNAS_NUMERICAS = preprocessing.COLUNAS_NUMERICAS
COLUNAS_CATEGORICAS = preprocessing.obter_colunas_categoricas(X_train)

PESO_CLASSE_POSITIVA = preprocessing.obter_scale_pos_weight(y_train)

COLUNAS_NUMERICAS, PESO_CLASSE_POSITIVA

(['IDADE_ANOS', 'SEMANA_EPI_SIN_PRI'], np.float64(31.894495412844037))

Confirma os valores default reais dos hiperparametros usados nas secoes seguintes, lendo-os diretamente da biblioteca instalada neste projeto, em vez de assumi-los de memoria.
Para Regressao Logistica, Arvore de Decisao, Random Forest e LightGBM, o proprio metodo `get_params()` de um estimador recem instanciado, sem argumentos, ja expoe o valor default de cada hiperparametro.
Para XGBoost, o wrapper do scikit-learn reporta `None` para hiperparametros nao definidos explicitamente, porque o valor default so e resolvido internamente no momento do ajuste, entao o estimador e ajustado sobre dados sinteticos minimos apenas para leitura da configuracao interna do booster apos o ajuste, sem qualquer relacao com os dados do estudo.
Esses valores confirmados sao usados a seguir para montar o `grid_padrao` de cada algoritmo, com um unico ponto por hiperparametro, representando a configuracao padrao da propria biblioteca.

In [5]:
gerador_dados_sinteticos = np.random.default_rng(42)
X_sintetico = gerador_dados_sinteticos.normal(size=(200, 4))
y_sintetico = gerador_dados_sinteticos.integers(0, 2, size=200)

modelo_xgboost_nao_ajustado = XGBClassifier()
modelo_xgboost_nao_ajustado.fit(X_sintetico, y_sintetico)
configuracao_interna_xgboost = json.loads(modelo_xgboost_nao_ajustado.get_booster().save_config())
parametros_arvore_xgboost = configuracao_interna_xgboost["learner"]["gradient_booster"]["tree_train_param"]

valores_default_confirmados = pd.DataFrame(
    [
        {
            "algoritmo": "regressao_logistica",
            "hiperparametro": "C",
            "valor_default": LogisticRegression().get_params()["C"],
        },
        {
            "algoritmo": "arvore_decisao",
            "hiperparametro": "max_depth",
            "valor_default": DecisionTreeClassifier().get_params()["max_depth"],
        },
        {
            "algoritmo": "arvore_decisao",
            "hiperparametro": "min_samples_leaf",
            "valor_default": DecisionTreeClassifier().get_params()["min_samples_leaf"],
        },
        {
            "algoritmo": "random_forest",
            "hiperparametro": "n_estimators",
            "valor_default": RandomForestClassifier().get_params()["n_estimators"],
        },
        {
            "algoritmo": "random_forest",
            "hiperparametro": "max_depth",
            "valor_default": RandomForestClassifier().get_params()["max_depth"],
        },
        {
            "algoritmo": "random_forest",
            "hiperparametro": "min_samples_leaf",
            "valor_default": RandomForestClassifier().get_params()["min_samples_leaf"],
        },
        {
            "algoritmo": "xgboost",
            "hiperparametro": "n_estimators",
            "valor_default": modelo_xgboost_nao_ajustado.get_booster().num_boosted_rounds(),
        },
        {
            "algoritmo": "xgboost",
            "hiperparametro": "max_depth",
            "valor_default": int(parametros_arvore_xgboost["max_depth"]),
        },
        {
            "algoritmo": "xgboost",
            "hiperparametro": "learning_rate",
            "valor_default": round(float(parametros_arvore_xgboost["learning_rate"]), 3),
        },
        {
            "algoritmo": "lightgbm",
            "hiperparametro": "n_estimators",
            "valor_default": LGBMClassifier().get_params()["n_estimators"],
        },
        {
            "algoritmo": "lightgbm",
            "hiperparametro": "max_depth",
            "valor_default": LGBMClassifier().get_params()["max_depth"],
        },
        {
            "algoritmo": "lightgbm",
            "hiperparametro": "learning_rate",
            "valor_default": LGBMClassifier().get_params()["learning_rate"],
        },
    ]
)

valores_default_confirmados

,algoritmo,hiperparametro,valor_default
0,regressao_logistica,C,1.0
1,arvore_decisao,max_depth,NaN
2,arvore_decisao,min_samples_leaf,1.0
3,random_forest,n_estimators,100.0
4,random_forest,max_depth,NaN
5,random_forest,min_samples_leaf,1.0
6,xgboost,n_estimators,100.0
7,xgboost,max_depth,6.0
8,xgboost,learning_rate,0.3
9,lightgbm,n_estimators,100.0


Define uma funcao auxiliar que executa o protocolo comum aos cinco algoritmos, agora comparando duas versoes de cada um.
A versao padrao usa o `grid_padrao`, uma grade de um unico ponto fixado nos valores default confirmados na celula anterior, e a versao otimizada usa o `grid_otimizado`, a mesma grade de busca ja usada neste notebook antes desta comparacao.
As duas versoes passam pelo mesmo `GridSearchCV`, com os mesmos tres folds de expanding window e a mesma metrica AUPRC (`average_precision`), e a versao selecionada e a de maior `best_score_`, a media da AUPRC nos tres folds, com a versao padrao vencendo em caso de empate exato, por parcimonia.
A partir da selecao, a funcao usa exclusivamente o `best_estimator_` da busca vencedora para treinar o modelo final, avaliar no teste de 2024, gerar as predicoes de teste de 2024 e de validacao de 2025, e salvar os artefatos correspondentes.
Como o `GridSearchCV` e criado com `refit=True`, que e o padrao, o proprio objeto retornado por `busca.fit` ja contem o modelo final treinado em todo o conjunto de treino com os melhores hiperparametros encontrados, sem necessidade de um treino separado.
Esta funcao e reutilizada pelas cinco secoes seguintes, para garantir que o protocolo de comparacao entre versao padrao e otimizada seja identico entre os algoritmos.

In [6]:
def executar_algoritmo(nome_algoritmo, pipeline, grid_padrao, grid_otimizado):
    busca_padrao = GridSearchCV(
        estimator=pipeline,
        param_grid=grid_padrao,
        scoring="average_precision",
        cv=folds_expanding_window,
        refit=True,
        n_jobs=-1,
    )
    busca_padrao.fit(X_train, y_train)

    busca_otimizada = GridSearchCV(
        estimator=pipeline,
        param_grid=grid_otimizado,
        scoring="average_precision",
        cv=folds_expanding_window,
        refit=True,
        n_jobs=-1,
    )
    busca_otimizada.fit(X_train, y_train)

    if busca_otimizada.best_score_ > busca_padrao.best_score_:
        busca_selecionada = busca_otimizada
        versao_selecionada = "otimizada"
    else:
        busca_selecionada = busca_padrao
        versao_selecionada = "padrao"

    preprocessing.salvar_resultados_grid_search(
        busca_selecionada, DIRETORIO_METRICAS / f"{nome_algoritmo}_grid_search.csv"
    )

    melhor_modelo = busca_selecionada.best_estimator_
    preprocessing.salvar_modelo(DIRETORIO_MODELOS / f"{nome_algoritmo}.joblib", melhor_modelo)

    probabilidade_teste2024 = melhor_modelo.predict_proba(X_test)[:, 1]
    preprocessing.salvar_predicoes(
        DIRETORIO_METRICAS / f"{nome_algoritmo}_predicoes_teste2024.parquet",
        probabilidade_teste2024,
        y_test,
    )

    probabilidade_val2025 = melhor_modelo.predict_proba(X_val2025)[:, 1]
    preprocessing.salvar_predicoes(
        DIRETORIO_METRICAS / f"{nome_algoritmo}_predicoes_val2025.parquet",
        probabilidade_val2025,
        y_val2025,
    )

    metricas_teste2024 = preprocessing.avaliar_predicoes(y_test, probabilidade_teste2024)
    metricas_teste2024["algoritmo"] = nome_algoritmo
    metricas_teste2024["melhores_hiperparametros"] = busca_selecionada.best_params_
    metricas_teste2024["versao_selecionada"] = versao_selecionada
    metricas_teste2024["auprc_media_cv_padrao"] = busca_padrao.best_score_
    metricas_teste2024["auprc_media_cv_otimizada"] = busca_otimizada.best_score_
    metricas_teste2024["hiperparametros_padrao"] = busca_padrao.best_params_
    metricas_teste2024["hiperparametros_otimizados"] = busca_otimizada.best_params_

    return metricas_teste2024


resultados_por_algoritmo = []

## Regressao Logistica

O pre-processamento usa codificacao one hot para as variaveis categoricas e padronizacao das variaveis numericas, ja que a Regressao Logistica e sensivel a escala dos preditores.
O desbalanceamento de classes e tratado por `class_weight="balanced"`, em ambas as versoes comparadas.
A versao padrao fixa a forca de regularizacao `C` no valor default confirmado da biblioteca, 1,0, e a versao otimizada varia esse mesmo hiperparametro numa grade mais ampla, ja que o conjunto de treino e pequeno e uma grade mais ampla aumentaria o risco de ajuste ao ruido da validacao.
A funcao `executar_algoritmo` treina as duas versoes e seleciona a de maior AUPRC media nos tres folds de expanding window.

In [7]:
preprocessador_logistica = preprocessing.construir_preprocessador_one_hot(
    COLUNAS_NUMERICAS, COLUNAS_CATEGORICAS, escalonar_numericas=True
)

pipeline_logistica = Pipeline(
    steps=[
        ("preprocessador", preprocessador_logistica),
        (
            "classificador",
            LogisticRegression(class_weight="balanced", max_iter=5000, random_state=42),
        ),
    ]
)

grid_logistica_padrao = {"classificador__C": [1.0]}
grid_logistica_otimizado = {"classificador__C": [0.01, 0.1, 1, 10]}

resultado_logistica = executar_algoritmo(
    "regressao_logistica", pipeline_logistica, grid_logistica_padrao, grid_logistica_otimizado
)
resultados_por_algoritmo.append(resultado_logistica)
resultado_logistica

{'auprc': 0.20660416542790572,
 'roc_auc': 0.8106330340746786,
 'brier_score': 0.23932031311927152,
 'algoritmo': 'regressao_logistica',
 'melhores_hiperparametros': {'classificador__C': 0.01},
 'versao_selecionada': 'otimizada',
 'auprc_media_cv_padrao': np.float64(0.07053396897725756),
 'auprc_media_cv_otimizada': np.float64(0.09452208867294681),
 'hiperparametros_padrao': {'classificador__C': 1.0},
 'hiperparametros_otimizados': {'classificador__C': 0.01}}

## Arvore de Decisao (CART)

O pre-processamento usa a mesma codificacao one hot das variaveis categoricas, sem padronizacao das numericas, ja que arvores de decisao nao dependem da escala dos preditores.
O desbalanceamento de classes tambem e tratado por `class_weight="balanced"`, em ambas as versoes comparadas.
A versao padrao fixa a profundidade maxima e o numero minimo de observacoes por folha nos valores default confirmados da biblioteca, sem limite de profundidade e uma observacao por folha, e a versao otimizada varia esses dois hiperparametros, os controles mais diretos de complexidade e de risco de sobreajuste numa unica arvore.
A funcao `executar_algoritmo` treina as duas versoes e seleciona a de maior AUPRC media nos tres folds de expanding window.

In [8]:
preprocessador_arvore_decisao = preprocessing.construir_preprocessador_one_hot(
    COLUNAS_NUMERICAS, COLUNAS_CATEGORICAS, escalonar_numericas=False
)

pipeline_arvore_decisao = Pipeline(
    steps=[
        ("preprocessador", preprocessador_arvore_decisao),
        ("classificador", DecisionTreeClassifier(class_weight="balanced", random_state=42)),
    ]
)

grid_arvore_decisao_padrao = {
    "classificador__max_depth": [None],
    "classificador__min_samples_leaf": [1],
}
grid_arvore_decisao_otimizado = {
    "classificador__max_depth": [3, 5, 7, None],
    "classificador__min_samples_leaf": [5, 10, 20],
}

resultado_arvore_decisao = executar_algoritmo(
    "arvore_decisao", pipeline_arvore_decisao, grid_arvore_decisao_padrao, grid_arvore_decisao_otimizado
)
resultados_por_algoritmo.append(resultado_arvore_decisao)
resultado_arvore_decisao

{'auprc': 0.12043299020034721,
 'roc_auc': 0.7148764282799429,
 'brier_score': 0.23922368554557322,
 'algoritmo': 'arvore_decisao',
 'melhores_hiperparametros': {'classificador__max_depth': 3,
  'classificador__min_samples_leaf': 5},
 'versao_selecionada': 'otimizada',
 'auprc_media_cv_padrao': np.float64(0.028006989770718842),
 'auprc_media_cv_otimizada': np.float64(0.05805949582773475),
 'hiperparametros_padrao': {'classificador__max_depth': None,
  'classificador__min_samples_leaf': 1},
 'hiperparametros_otimizados': {'classificador__max_depth': 3,
  'classificador__min_samples_leaf': 5}}

## Random Forest

O pre-processamento e identico ao da Arvore de Decisao, com codificacao one hot e sem padronizacao das numericas.
O desbalanceamento de classes e tratado por `class_weight="balanced"`, aplicado dentro de cada arvore do ensemble, em ambas as versoes comparadas.
A versao padrao fixa o numero de arvores, a profundidade maxima e o numero minimo de observacoes por folha nos valores default confirmados da biblioteca, cem arvores, sem limite de profundidade e uma observacao por folha, e a versao otimizada varia esses tres hiperparametros numa grade mais ampla.
A funcao `executar_algoritmo` treina as duas versoes e seleciona a de maior AUPRC media nos tres folds de expanding window.

In [9]:
preprocessador_random_forest = preprocessing.construir_preprocessador_one_hot(
    COLUNAS_NUMERICAS, COLUNAS_CATEGORICAS, escalonar_numericas=False
)

pipeline_random_forest = Pipeline(
    steps=[
        ("preprocessador", preprocessador_random_forest),
        (
            "classificador",
            RandomForestClassifier(class_weight="balanced", random_state=42, n_jobs=1),
        ),
    ]
)

grid_random_forest_padrao = {
    "classificador__n_estimators": [100],
    "classificador__max_depth": [None],
    "classificador__min_samples_leaf": [1],
}
grid_random_forest_otimizado = {
    "classificador__n_estimators": [200, 500],
    "classificador__max_depth": [5, 10, None],
    "classificador__min_samples_leaf": [1, 5],
}

resultado_random_forest = executar_algoritmo(
    "random_forest", pipeline_random_forest, grid_random_forest_padrao, grid_random_forest_otimizado
)
resultados_por_algoritmo.append(resultado_random_forest)
resultado_random_forest

{'auprc': 0.18700812278617998,
 'roc_auc': 0.8049658233013671,
 'brier_score': 0.07264001245363054,
 'algoritmo': 'random_forest',
 'melhores_hiperparametros': {'classificador__max_depth': None,
  'classificador__min_samples_leaf': 5,
  'classificador__n_estimators': 500},
 'versao_selecionada': 'otimizada',
 'auprc_media_cv_padrao': np.float64(0.05566779047603699),
 'auprc_media_cv_otimizada': np.float64(0.08134569598216673),
 'hiperparametros_padrao': {'classificador__max_depth': None,
  'classificador__min_samples_leaf': 1,
  'classificador__n_estimators': 100},
 'hiperparametros_otimizados': {'classificador__max_depth': None,
  'classificador__min_samples_leaf': 5,
  'classificador__n_estimators': 500}}

## XGBoost

O pre-processamento usa o imputador categorico nativo de `utils/preprocessing.py`, que preenche ausentes e expoe as variaveis categoricas como tipo `category`, para que o XGBoost use sua propria codificacao categorica em vez de codificacao one hot.
O desbalanceamento de classes e tratado por `scale_pos_weight`, calculado a partir da razao entre casos sem obito e com obito no treino completo, em ambas as versoes comparadas.
A versao padrao fixa o numero de arvores, a profundidade maxima e a taxa de aprendizado nos valores default confirmados da biblioteca, cem arvores, profundidade maxima seis e taxa de aprendizado 0,3, e a versao otimizada varia esses tres hiperparametros numa grade mais conservadora.
A funcao `executar_algoritmo` treina as duas versoes e seleciona a de maior AUPRC media nos tres folds de expanding window.

In [10]:
preprocessador_xgboost = preprocessing.ImputadorCategoricoNativo(COLUNAS_NUMERICAS, COLUNAS_CATEGORICAS)

pipeline_xgboost = Pipeline(
    steps=[
        ("preprocessador", preprocessador_xgboost),
        (
            "classificador",
            XGBClassifier(
                objective="binary:logistic",
                eval_metric="aucpr",
                tree_method="hist",
                enable_categorical=True,
                scale_pos_weight=PESO_CLASSE_POSITIVA,
                random_state=42,
                n_jobs=1,
            ),
        ),
    ]
)

grid_xgboost_padrao = {
    "classificador__n_estimators": [100],
    "classificador__max_depth": [6],
    "classificador__learning_rate": [0.3],
}
grid_xgboost_otimizado = {
    "classificador__n_estimators": [100, 300],
    "classificador__max_depth": [3, 5],
    "classificador__learning_rate": [0.05, 0.1],
}

resultado_xgboost = executar_algoritmo(
    "xgboost", pipeline_xgboost, grid_xgboost_padrao, grid_xgboost_otimizado
)
resultados_por_algoritmo.append(resultado_xgboost)
resultado_xgboost

{'auprc': 0.19624488287305888,
 'roc_auc': 0.7659304223627832,
 'brier_score': 0.1896233707666397,
 'algoritmo': 'xgboost',
 'melhores_hiperparametros': {'classificador__learning_rate': 0.05,
  'classificador__max_depth': 3,
  'classificador__n_estimators': 100},
 'versao_selecionada': 'otimizada',
 'auprc_media_cv_padrao': np.float64(0.06327749750908444),
 'auprc_media_cv_otimizada': np.float64(0.06650505618767404),
 'hiperparametros_padrao': {'classificador__learning_rate': 0.3,
  'classificador__max_depth': 6,
  'classificador__n_estimators': 100},
 'hiperparametros_otimizados': {'classificador__learning_rate': 0.05,
  'classificador__max_depth': 3,
  'classificador__n_estimators': 100}}

## LightGBM

O pre-processamento usa o mesmo imputador categorico nativo do XGBoost, ja que o LightGBM tambem reconhece diretamente colunas do tipo `category`.
O desbalanceamento de classes tambem e tratado por `scale_pos_weight`, em ambas as versoes comparadas.
A versao padrao fixa o numero de arvores, a profundidade maxima e a taxa de aprendizado nos valores default confirmados da biblioteca, cem arvores, sem limite de profundidade e taxa de aprendizado 0,1, e a versao otimizada varia esses tres hiperparametros na mesma escala usada para o XGBoost, para facilitar a comparacao entre os dois algoritmos baseados em gradient boosting.
A funcao `executar_algoritmo` treina as duas versoes e seleciona a de maior AUPRC media nos tres folds de expanding window.

In [11]:
preprocessador_lightgbm = preprocessing.ImputadorCategoricoNativo(COLUNAS_NUMERICAS, COLUNAS_CATEGORICAS)

pipeline_lightgbm = Pipeline(
    steps=[
        ("preprocessador", preprocessador_lightgbm),
        (
            "classificador",
            LGBMClassifier(
                objective="binary",
                scale_pos_weight=PESO_CLASSE_POSITIVA,
                random_state=42,
                n_jobs=1,
                verbosity=-1,
            ),
        ),
    ]
)

grid_lightgbm_padrao = {
    "classificador__n_estimators": [100],
    "classificador__max_depth": [-1],
    "classificador__learning_rate": [0.1],
}
grid_lightgbm_otimizado = {
    "classificador__n_estimators": [100, 300],
    "classificador__max_depth": [3, 5, -1],
    "classificador__learning_rate": [0.05, 0.1],
}

resultado_lightgbm = executar_algoritmo(
    "lightgbm", pipeline_lightgbm, grid_lightgbm_padrao, grid_lightgbm_otimizado
)
resultados_por_algoritmo.append(resultado_lightgbm)
resultado_lightgbm

{'auprc': 0.16829753416354531,
 'roc_auc': 0.7455863599265455,
 'brier_score': 0.1403487511859102,
 'algoritmo': 'lightgbm',
 'melhores_hiperparametros': {'classificador__learning_rate': 0.1,
  'classificador__max_depth': 3,
  'classificador__n_estimators': 300},
 'versao_selecionada': 'otimizada',
 'auprc_media_cv_padrao': np.float64(0.08158055726578024),
 'auprc_media_cv_otimizada': np.float64(0.11294571103824946),
 'hiperparametros_padrao': {'classificador__learning_rate': 0.1,
  'classificador__max_depth': -1,
  'classificador__n_estimators': 100},
 'hiperparametros_otimizados': {'classificador__learning_rate': 0.1,
  'classificador__max_depth': 3,
  'classificador__n_estimators': 300}}

## Comparacao consolidada dos cinco algoritmos

Reune as metricas de teste de 2024 dos cinco algoritmos numa unica tabela, ordenada pela AUPRC em ordem decrescente, a metrica mais informativa neste problema dada a baixa prevalencia de obito.
A tabela tambem registra a versao selecionada de cada algoritmo, padrao ou otimizada, e os hiperparametros correspondentes a essa versao, para referencia.
O resultado e salvo em `analysis/results_modelagem/comparacao_modelos_teste2024.csv`.

In [12]:
tabela_comparativa = pd.DataFrame(resultados_por_algoritmo)
tabela_comparativa = (
    tabela_comparativa[
        ["algoritmo", "auprc", "roc_auc", "brier_score", "versao_selecionada", "melhores_hiperparametros"]
    ]
    .sort_values("auprc", ascending=False)
    .reset_index(drop=True)
)

tabela_comparativa.to_csv(DIRETORIO_RESULTADOS_ANALISE / "comparacao_modelos_teste2024.csv", index=False)
tabela_comparativa

,algoritmo,auprc,roc_auc,brier_score,versao_selecionada,melhores_hiperparametros
0,regressao_logistica,0.206604,0.810633,0.239320,otimizada,{'classificador__C': 0.01}
1,xgboost,0.196245,0.765930,0.189623,otimizada,"{'classificador__learning_rate': 0.05, 'classi..."
2,random_forest,0.187008,0.804966,0.072640,otimizada,"{'classificador__max_depth': None, 'classifica..."
3,lightgbm,0.168298,0.745586,0.140349,otimizada,"{'classificador__learning_rate': 0.1, 'classif..."
4,arvore_decisao,0.120433,0.714876,0.239224,otimizada,"{'classificador__max_depth': 3, 'classificador..."


Consolida, para cada um dos cinco algoritmos, a comparacao entre a versao padrao e a versao otimizada que fundamentou a selecao feita dentro de `executar_algoritmo`.
A tabela reune a AUPRC media nos tres folds de expanding window de cada versao, a versao efetivamente selecionada, e os hiperparametros de cada uma das duas versoes.
Esta tabela e o registro oficial de qual versao de cada algoritmo segue para as fases posteriores do estudo, como calibracao, avaliacao, importancia de variaveis e equidade, e e salva em `output/metricas/selecao_versao_modelos.csv`.

In [13]:
tabela_selecao_versao = pd.DataFrame(
    [
        {
            "algoritmo": resultado["algoritmo"],
            "auprc_media_cv_padrao": resultado["auprc_media_cv_padrao"],
            "auprc_media_cv_otimizada": resultado["auprc_media_cv_otimizada"],
            "versao_selecionada": resultado["versao_selecionada"],
            "hiperparametros_padrao": resultado["hiperparametros_padrao"],
            "hiperparametros_otimizados": resultado["hiperparametros_otimizados"],
        }
        for resultado in resultados_por_algoritmo
    ]
)

tabela_selecao_versao.to_csv(DIRETORIO_METRICAS / "selecao_versao_modelos.csv", index=False)
tabela_selecao_versao

,algoritmo,auprc_media_cv_padrao,auprc_media_cv_otimizada,versao_selecionada,hiperparametros_padrao,hiperparametros_otimizados
0,regressao_logistica,0.070534,0.094522,otimizada,{'classificador__C': 1.0},{'classificador__C': 0.01}
1,arvore_decisao,0.028007,0.058059,otimizada,"{'classificador__max_depth': None, 'classifica...","{'classificador__max_depth': 3, 'classificador..."
2,random_forest,0.055668,0.081346,otimizada,"{'classificador__max_depth': None, 'classifica...","{'classificador__max_depth': None, 'classifica..."
3,xgboost,0.063277,0.066505,otimizada,"{'classificador__learning_rate': 0.3, 'classif...","{'classificador__learning_rate': 0.05, 'classi..."
4,lightgbm,0.081581,0.112946,otimizada,"{'classificador__learning_rate': 0.1, 'classif...","{'classificador__learning_rate': 0.1, 'classif..."


## Sintese dos achados

### Comparacao entre versao padrao e versao otimizada

Esta secao incorpora, para cada um dos cinco algoritmos, a comparacao entre a versao padrao, com os hiperparametros default da biblioteca instalada neste projeto, e a versao otimizada, ajustada por busca em grade, pela AUPRC media nos tres folds de expanding window do treino.
Nos cinco algoritmos, a versao otimizada obteve AUPRC media de validacao cruzada maior do que a versao padrao, sem nenhum empate, e por isso a versao otimizada foi a selecionada em todos os casos.
A Regressao Logistica teve AUPRC media de 0,071 na versao padrao e 0,095 na versao otimizada.
A Arvore de Decisao teve 0,028 na versao padrao e 0,058 na versao otimizada.
O Random Forest teve 0,056 na versao padrao e 0,081 na versao otimizada.
O XGBoost teve 0,063 na versao padrao e 0,067 na versao otimizada, a menor diferenca relativa entre as duas versoes dentre os cinco algoritmos.
O LightGBM teve 0,082 na versao padrao e 0,113 na versao otimizada.
Como a versao otimizada venceu nos cinco algoritmos, o modelo final de cada algoritmo, os hiperparametros selecionados e as metricas de teste de 2024 sao identicos aos valores ja documentados neste notebook antes desta comparacao: Regressao Logistica com AUPRC 0,207 e ROC-AUC 0,811, XGBoost com AUPRC 0,196 e ROC-AUC 0,766, Random Forest com AUPRC 0,187 e ROC-AUC 0,805, LightGBM com AUPRC 0,168 e ROC-AUC 0,746, e Arvore de Decisao com AUPRC 0,120 e ROC-AUC 0,715.
Nenhuma metrica de teste de 2024 mudou em relacao aos valores documentados antes desta comparacao, para nenhum dos cinco algoritmos, porque em nenhum deles a versao padrao superou a versao otimizada.
Vale notar que a AUPRC media de validacao cruzada, tanto da versao padrao quanto da versao otimizada, e sistematicamente bem mais baixa do que a AUPRC observada depois no teste de 2024, nos cinco algoritmos.
Essa diferenca e esperada dado o tamanho dos folds de expanding window, com o primeiro fold treinando em 2020 e validando em apenas 2021, um unico ano com poucos obitos, o que torna a AUPRC de validacao cruzada mais sensivel a ruido amostral do que a AUPRC calculada sobre o teste de 2024 completo, um conjunto maior e usado uma unica vez.
Essa comparacao entre versao padrao e versao otimizada usa exclusivamente os tres folds de expanding window do treino, de 2020 a 2023, nunca o teste de 2024 ou a validacao externa de 2025, preservando a validacao temporal do projeto.

### Leitura das metricas de teste de 2024

A comparacao no teste de 2024 mostra que os cinco algoritmos superam amplamente o que se esperaria de um classificador sem poder discriminativo, cuja AUPRC esperada seria proxima da prevalencia observada de obito nesse ano, 5,23 por cento.
A Regressao Logistica obteve a melhor AUPRC entre os cinco algoritmos, 0,207, e tambem o melhor ROC-AUC, 0,811, mas teve o pior Brier Score entre os cinco, 0,239, o que e consistente com o uso de class_weight="balanced", que melhora a capacidade de discriminar os casos de maior risco mas desloca as probabilidades previstas para valores sistematicamente mais altos do que a prevalencia real.
Esse Brier Score e praticamente indistinguivel do da Arvore de Decisao, tambem 0,239, a diferenca entre os dois aparece somente na quarta casa decimal.
O XGBoost obteve a segunda melhor AUPRC, 0,196, e o terceiro melhor ROC-AUC, 0,766, selecionando o menor numero de arvores, a menor profundidade maxima e a menor taxa de aprendizado disponiveis na grade otimizada, uma combinacao bem mais conservadora do que a escolha dos valores mais altos da mesma grade, e tambem mais conservadora do que a propria versao padrao, que usa profundidade maxima seis e taxa de aprendizado 0,3.
O Random Forest obteve a terceira melhor AUPRC, 0,187, e o segundo melhor ROC-AUC, 0,805, mas foi o algoritmo com o melhor Brier Score entre os cinco, 0,073, uma combinacao que indica probabilidades absolutas bem calibradas apesar de uma capacidade de ordenacao dos casos de maior risco um pouco inferior a da Regressao Logistica e do XGBoost.
O LightGBM obteve a quarta melhor AUPRC, 0,168, e o quarto melhor ROC-AUC, 0,746, selecionando a maior taxa de aprendizado e o maior numero de arvores disponiveis na grade otimizada, com a profundidade maxima mais restritiva entre as tres opcoes testadas.
A Arvore de Decisao teve tanto a AUPRC quanto o ROC-AUC mais baixos entre os cinco algoritmos, 0,120 e 0,715 respectivamente, mas seu Brier Score, 0,239, nao foi o mais alto, como visto acima esse valor fica ligeiramente abaixo do da Regressao Logistica.
Isso mostra que nenhuma metrica isolada captura toda a comparacao entre os cinco algoritmos, e que uma boa ordenacao dos casos de maior risco, medida pela AUPRC e pelo ROC-AUC, nao garante por si so probabilidades absolutas bem calibradas.
Essa dissociacao entre ordenacao e calibracao e consistente com o volume reduzido de obitos no treino, 7171 casos com apenas 218 obitos, que torna tanto a escolha de hiperparametros quanto a estimativa das proprias metricas mais sensiveis a ruido amostral.
Esta comparacao usa exclusivamente o teste de 2024, conforme a validacao temporal do projeto, e as predicoes sobre a validacao externa de 2025 foram geradas e salvas para os cinco algoritmos, mas nao influenciaram a leitura comparativa feita aqui.
O numero reduzido de obitos em cada conjunto, 218 no treino e 208 no teste de 2024, limita a precisao de todas as metricas estimadas neste notebook, e a quantificacao dessa incerteza por meio de intervalos de confianca fica para uma fase posterior de avaliacao.